# Phase 2: CNN Model Development & Evaluation
**Project:** Concrete Vision — Automated Visual Inspection System  
**Framework:** PyTorch  
**Input:** Greyscale 256×256 images (1-channel) — justified in Phase 1  
**Manifests:** Load from `manifests/` folder produced in Phase 1

---

## Notebook Structure
1. [Setup & Configuration](#1-setup--configuration)
2. [Data Pipeline](#2-data-pipeline)
3. [Class Weights & Metrics](#3-class-weights--metrics)
4. [Model Architecture](#4-model-architecture)
5. [Training](#5-training)
6. [Threshold Calibration](#6-threshold-calibration)
7. [Test Set Evaluation](#7-test-set-evaluation)
8. [Cross-Domain Diagnostic](#8-cross-domain-diagnostic)


## 1. Setup & Configuration

In [1]:
import os, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (confusion_matrix, recall_score, precision_score,
                             f1_score, roc_auc_score, precision_recall_curve)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH    = Path("..").resolve()
MANIFEST_DIR = BASE_PATH / "manifests"
CHECKPOINT   = BASE_PATH / "checkpoints"
CHECKPOINT.mkdir(exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE     = 256
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR           = 1e-3
PATIENCE     = 7       # early stopping patience (epochs)


Device: cpu


**Configuration notes:**
- `SEED = 42` fixed across all libraries for full reproducibility
- `PATIENCE = 7` — if validation recall does not improve for 7 consecutive epochs, training stops and the best checkpoint is restored
- `BATCH_SIZE = 32` — suitable for a local GPU with ≥4GB VRAM; reduce to 16 if memory errors occur

## 2. Data Pipeline

### 2.1 Augmentation Strategy

Augmentation is applied to the **training set only**. Choices are grounded in Phase 1 findings:

| Transform | Justification |
|---|---|
| Horizontal & vertical flip | Cracks have no canonical orientation |
| Random rotation ±15° | Crack direction is arbitrary |
| Brightness & contrast jitter | Phase 1 showed wide luminance variation across domains |
| Gaussian blur (small) | Simulates varying camera focus in drone footage |
| Normalise to [0, 1] | Required for numerical stability — see Phase 1 decision |

Zoom/crop was **excluded**: at 256×256 with thin hairline cracks, aggressive cropping risks removing the crack entirely, corrupting the label.

In [2]:
# ── Transforms ───────────────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),           # converts to [0, 1] automatically
])

val_test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])


### 2.2 Dataset Class

In [3]:
class CrackDataset(Dataset):
    """PyTorch Dataset for crack detection.
    Loads images from filepaths listed in a manifest CSV.
    Labels: Cracked = 1, Non-Cracked = 0.
    """
    LABEL_MAP = {"Cracked": 1, "Non-Cracked": 0}

    def __init__(self, manifest: pd.DataFrame, transform=None):
        self.df        = manifest.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row["filepath"]).convert("RGB")  # convert to RGB then grayscale via transform
        label = self.LABEL_MAP[row["label"]]
        if self.transform:
            img = self.transform(img)
        return img, label, row["filepath"]   # return filepath for flagged-image tracking


### 2.3 Load Manifests & Build DataLoaders

In [4]:
# ── Load manifests ────────────────────────────────────────────────────────────
df_train = pd.read_csv(MANIFEST_DIR / "train_manifest.csv")
df_val   = pd.read_csv(MANIFEST_DIR / "val_manifest.csv")
df_test  = pd.read_csv(MANIFEST_DIR / "test_manifest.csv")
df_flagged = pd.read_csv(MANIFEST_DIR / "flagged_images_manifest.csv")

print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")
print(f"Flagged subset: {len(df_flagged):,} images")
print(f"\nTrain class balance:")
print(df_train["label"].value_counts().to_string())


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\id222\\OneDrive - Imperial College London\\Documents\\GitHub\\manifests\\train_manifest.csv'

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_ds = CrackDataset(df_train, transform=train_transforms)
val_ds   = CrackDataset(df_val,   transform=val_test_transforms)
test_ds  = CrackDataset(df_test,  transform=val_test_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}  "
      f"Val batches: {len(val_loader)}  "
      f"Test batches: {len(test_loader)}")


## 3. Class Weights & Metrics

### 3.1 Class Weights

The training set is severely imbalanced (~1:4 Cracked:Non-Cracked for Walls, ~1:6 for Decks). Class weights increase the loss penalty for misclassifying a Cracked image, directly pushing the model toward higher recall.

Weight formula: `w_cracked = n_total / (2 × n_cracked)`

In [ ]:
n_total   = len(df_train)
n_cracked = (df_train["label"] == "Cracked").sum()
n_noncracked = n_total - n_cracked

w_cracked    = n_total / (2 * n_cracked)
w_noncracked = n_total / (2 * n_noncracked)

print(f"Cracked:     {n_cracked:,}  → weight {w_cracked:.3f}")
print(f"Non-Cracked: {n_noncracked:,}  → weight {w_noncracked:.3f}")

# Tensor for BCEWithLogitsLoss pos_weight argument
# pos_weight = ratio of negative to positive samples
pos_weight = torch.tensor([n_noncracked / n_cracked], dtype=torch.float32).to(DEVICE)
print(f"\npos_weight (passed to loss function): {pos_weight.item():.3f}")


### 3.2 Metric Definitions

| Metric | Role | Rationale |
|---|---|---|
| **Recall (Sensitivity)** | Primary | A missed crack is a safety failure — this is the metric we optimise and report first |
| **F2-Score** | Secondary | Weights recall twice as heavily as precision; appropriate for safety-critical triage |
| **Precision** | Tertiary | Controls false positive rate — too many false positives makes the triage system unworkable |
| **AUC-ROC** | Tertiary | Threshold-independent overall discriminative power |
| **Confusion Matrix** | Reporting | Reported per domain (Walls, Decks) to reveal surface-type-specific failure modes |

> **Accuracy is not reported as a primary metric.** With ~80% Non-Cracked images, a model predicting Non-Cracked for everything achieves ~80% accuracy while having zero recall — exactly the failure mode the brief warns against.

In [ ]:
def f2_score(y_true, y_pred):
    """F2 score — weights recall twice as heavily as precision."""
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    if p + r == 0:
        return 0.0
    return (1 + 2**2) * p * r / (2**2 * p + r)

def compute_metrics(y_true, y_pred_binary, y_pred_prob):
    """Return a dict of all metrics given true labels, binary predictions, and raw probabilities."""
    return {
        "Recall":    recall_score(y_true, y_pred_binary, zero_division=0),
        "Precision": precision_score(y_true, y_pred_binary, zero_division=0),
        "F2":        f2_score(y_true, y_pred_binary),
        "AUC-ROC":   roc_auc_score(y_true, y_pred_prob),
    }

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix", ax=None):
    cm = confusion_matrix(y_true, y_pred)
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-Cracked", "Cracked"],
                yticklabels=["Non-Cracked", "Cracked"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(title, fontweight="bold")
    tn, fp, fn, tp = cm.ravel()
    ax.set_xlabel(f"Predicted  |  TP={tp} FP={fp} FN={fn} TN={tn}")


## 4. Model Architecture

### Design Decisions

| Choice | Rationale |
|---|---|
| **4 conv blocks** | Deep enough to learn crack morphology (thin, linear, high-aspect-ratio features); not so deep it overfits on ~5,500 cracked training images |
| **Filter depth 32→64→128→256** | Progressive feature abstraction from low-level edges to high-level crack patterns |
| **BatchNorm after every conv** | Stabilises training across the luminance variation documented in Phase 1 |
| **GlobalAveragePooling** | Fewer parameters than Flatten, better generalisation, less overfitting on minority class |
| **Dropout(0.5)** | Regularisation before the final classifier |
| **Single sigmoid output** | Binary classification — outputs probability of Cracked |

In [ ]:
class CrackCNN(nn.Module):
    """
    4-block CNN for binary crack detection.
    Input:  (B, 1, 256, 256) greyscale images, normalised to [0, 1]
    Output: (B, 1) sigmoid probability of Cracked
    """
    def __init__(self, dropout=0.5):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),          # halves spatial dimensions
            )

        self.features = nn.Sequential(
            conv_block(1,   32),   # 256 → 128
            conv_block(32,  64),   # 128 → 64
            conv_block(64,  128),  # 64  → 32
            conv_block(128, 256),  # 32  → 16
        )
        self.pool       = nn.AdaptiveAvgPool2d(1)   # GlobalAveragePooling → (B, 256, 1, 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x   # raw logits — sigmoid applied in loss and at inference

model = CrackCNN().to(DEVICE)

# Parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
print(model)


## 5. Training

In [ ]:
# ── Loss, optimiser, scheduler ───────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, verbose=True)
    # mode="max" because we're monitoring recall (higher is better)


In [ ]:
def run_epoch(loader, model, criterion, optimizer=None, threshold=0.5):
    """
    Run one epoch of training or evaluation.
    If optimizer is None, runs in eval mode (no gradient updates).
    Returns: avg loss, recall, all true labels, all predicted probs, all filepaths
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_labels, all_probs, all_paths = 0.0, [], [], []

    with torch.set_grad_enabled(is_train):
        for imgs, labels, paths in loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.float().to(DEVICE)

            logits = model(imgs).squeeze(1)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(imgs)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_paths.extend(paths)

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    preds      = (all_probs >= threshold).astype(int)
    avg_loss   = total_loss / len(loader.dataset)
    recall     = recall_score(all_labels, preds, zero_division=0)

    return avg_loss, recall, all_labels, all_probs, all_paths


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_recall": [], "val_recall": []}
best_val_recall = 0.0
epochs_no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_recall, _, _, _ = run_epoch(train_loader, model, criterion, optimizer)
    vl_loss, vl_recall, _, _, _ = run_epoch(val_loader,   model, criterion)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_recall"].append(tr_recall)
    history["val_recall"].append(vl_recall)

    scheduler.step(vl_recall)

    # ── Checkpoint on best validation recall ──────────────────────────────────
    if vl_recall > best_val_recall:
        best_val_recall = vl_recall
        epochs_no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT / "best_model.pt")
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
          f"Train Loss: {tr_loss:.4f}  Train Recall: {tr_recall:.3f}  "
          f"Val Loss: {vl_loss:.4f}  Val Recall: {vl_recall:.3f}"
          + ("  ← best" if epochs_no_improve == 0 else ""))

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} — no improvement for {PATIENCE} epochs.")
        break

print(f"\nBest validation recall: {best_val_recall:.4f}")


In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_ran, history["train_loss"], label="Train")
axes[0].plot(epochs_ran, history["val_loss"],   label="Val")
axes[0].set_title("Loss per Epoch", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE Loss")
axes[0].legend()

axes[1].plot(epochs_ran, history["train_recall"], label="Train")
axes[1].plot(epochs_ran, history["val_recall"],   label="Val")
axes[1].set_title("Recall per Epoch", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Recall")
axes[1].legend()

plt.suptitle("Training History", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


**What to look for in the training curves:**
- **Loss:** Both train and val loss should decrease. If val loss increases while train loss decreases, the model is overfitting — increase dropout or reduce model depth.
- **Recall:** Val recall should broadly track train recall. A large gap indicates overfitting. If both are low, the class weights may need increasing or the learning rate adjusting.
- **Early stopping:** The best checkpoint (saved to `checkpoints/best_model.pt`) is the epoch with highest val recall, not the final epoch.

## 6. Threshold Calibration

The default threshold of 0.5 is not appropriate for a safety-critical triage system. We use the validation set to find a threshold that maximises recall at acceptable precision.

**Target:** Recall ≥ 0.90 on the validation set. This means at most 1 in 10 cracked images is missed. The brief frames this as a triage system — human engineers review flagged images, so false positives are an inconvenience, not a safety failure. False negatives (missed cracks) are the safety risk.

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CHECKPOINT / "best_model.pt", map_location=DEVICE))
model.eval()
print("Best checkpoint loaded.")

# ── Get val set predictions ────────────────────────────────────────────────────
_, _, val_labels, val_probs, val_paths = run_epoch(val_loader, model, criterion)


In [ ]:
# ── Precision-Recall curve ────────────────────────────────────────────────────
precisions, recalls, thresholds = precision_recall_curve(val_labels, val_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve
axes[0].plot(recalls, precisions, color="#2980b9", linewidth=2)
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve (Validation)", fontweight="bold")
axes[0].axvline(0.90, color="#c0392b", linestyle="--", label="Recall = 0.90 target")
axes[0].legend()

# Recall & Precision vs Threshold
axes[1].plot(thresholds, recalls[:-1],    label="Recall",    color="#c0392b", linewidth=2)
axes[1].plot(thresholds, precisions[:-1], label="Precision", color="#2980b9", linewidth=2)
axes[1].axhline(0.90, color="grey", linestyle="--", linewidth=0.8)
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Score")
axes[1].set_title("Recall & Precision vs. Threshold (Validation)", fontweight="bold")
axes[1].legend()

plt.suptitle("Threshold Calibration", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Select threshold at recall ≥ 0.90 ────────────────────────────────────────
RECALL_TARGET = 0.90

# Find the highest threshold that still achieves recall >= target
valid_thresholds = thresholds[recalls[:-1] >= RECALL_TARGET]
if len(valid_thresholds) == 0:
    print("⚠️  Target recall not achieved — using threshold that maximises recall.")
    OPTIMAL_THRESHOLD = thresholds[np.argmax(recalls[:-1])]
else:
    OPTIMAL_THRESHOLD = valid_thresholds[-1]   # highest threshold (best precision) at target recall

val_preds_calibrated = (val_probs >= OPTIMAL_THRESHOLD).astype(int)
val_metrics = compute_metrics(val_labels, val_preds_calibrated, val_probs)

print(f"Optimal threshold: {OPTIMAL_THRESHOLD:.3f}")
print(f"\nValidation Metrics at Calibrated Threshold:")
for k, v in val_metrics.items():
    print(f"  {k:<12} {v:.4f}")


## 7. Test Set Evaluation

We now apply the calibrated threshold to the **held-out test set** — data the model has never seen during training or threshold selection. Results are reported overall and broken down by domain to reveal surface-type-specific performance.

In [ ]:
# ── Full test set evaluation ──────────────────────────────────────────────────
_, _, test_labels, test_probs, test_paths = run_epoch(test_loader, model, criterion)
test_preds = (test_probs >= OPTIMAL_THRESHOLD).astype(int)

test_metrics = compute_metrics(test_labels, test_preds, test_probs)
print("Test Set Metrics:")
for k, v in test_metrics.items():
    print(f"  {k:<12} {v:.4f}")


In [ ]:
# ── Per-domain confusion matrices ─────────────────────────────────────────────
# Rebuild test dataframe with predictions
df_test_results = df_test.copy().reset_index(drop=True)
df_test_results["true_label"] = test_labels.astype(int)
df_test_results["pred_label"] = test_preds
df_test_results["prob"]       = test_probs

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall
plot_confusion_matrix(test_labels, test_preds, title="Overall", ax=axes[0])

# Per domain
for ax, domain in zip(axes[1:], ["Walls", "Decks"]):
    sub = df_test_results[df_test_results["domain"] == domain]
    plot_confusion_matrix(sub["true_label"], sub["pred_label"],
                          title=f"{domain}", ax=ax)

plt.suptitle(f"Confusion Matrices — Test Set (threshold={OPTIMAL_THRESHOLD:.2f})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Per-domain metric summary ─────────────────────────────────────────────────
rows = []
for domain in ["Walls", "Decks", "Overall"]:
    sub = df_test_results if domain == "Overall" else df_test_results[df_test_results["domain"] == domain]
    m   = compute_metrics(sub["true_label"], sub["pred_label"], sub["prob"])
    rows.append({"Domain": domain, **m})

df_metrics = pd.DataFrame(rows).set_index("Domain").round(4)
print(df_metrics.to_string())


In [ ]:
# ── Flagged-image recall ──────────────────────────────────────────────────────
# Track recall separately on the hard confounder subset from Phase 1
flagged_paths = set(df_flagged["filepath"].values)
df_flagged_results = df_test_results[df_test_results["filepath"].isin(flagged_paths)]

if len(df_flagged_results) > 0:
    flagged_recall = recall_score(df_flagged_results["true_label"],
                                  df_flagged_results["pred_label"], zero_division=0)
    print(f"Flagged-image subset (n={len(df_flagged_results)}):")
    print(f"  Recall: {flagged_recall:.4f}")
    print(f"  (vs overall test recall: {test_metrics['Recall']:.4f})")
else:
    print("No flagged images found in test set — all may be in train split.")


### 7.1 Evaluation Findings

*(Complete after running the cells above.)*

Report the following:
- Overall recall, F2, precision, AUC-ROC at the calibrated threshold
- Whether Walls or Decks recall is lower and why (refer to Phase 1 domain differences)
- Whether flagged-image recall is lower than overall recall — if so, the model is struggling with the confounder cases identified in Phase 1
- Any systematic pattern in the confusion matrix (e.g. FN concentrated in one domain)

## 8. Cross-Domain Diagnostic

We train two additional models:
- **B1:** Train on Walls only → evaluate on Decks only
- **B2:** Train on Decks only → evaluate on Walls only

If recall collapses in cross-domain evaluation, this is evidence the model is learning **surface texture** rather than **crack morphology** — consistent with the partial domain separability observed in the Phase 1 PCA.

This directly informs the client recommendation: single generic model vs. domain-specific models.

> These runs use the same architecture and hyperparameters. Training is capped at 20 epochs since the domain-specific datasets are smaller.

In [ ]:
def build_domain_loaders(train_domain, test_domain):
    """Build train/test loaders for a single domain each."""
    df_tr = df_train[df_train["domain"] == train_domain]
    df_te = df_test[df_test["domain"]   == test_domain]

    # Recompute pos_weight for this domain's imbalance
    n_cr   = (df_tr["label"] == "Cracked").sum()
    n_ncr  = len(df_tr) - n_cr
    pw     = torch.tensor([n_ncr / n_cr], dtype=torch.float32).to(DEVICE)

    tr_loader = DataLoader(CrackDataset(df_tr, train_transforms),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    te_loader = DataLoader(CrackDataset(df_te, val_test_transforms),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    return tr_loader, te_loader, pw


def cross_domain_run(train_domain, test_domain, epochs=20):
    """Train on train_domain, evaluate on test_domain. Returns test recall and F2."""
    print(f"\n── Train: {train_domain}  →  Test: {test_domain} ──")
    tr_loader, te_loader, pw = build_domain_loaders(train_domain, test_domain)

    m   = CrackCNN().to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt  = optim.Adam(m.parameters(), lr=LR)

    best_recall, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        run_epoch(tr_loader, m, crit, opt)
        _, vl_recall, _, _, _ = run_epoch(te_loader, m, crit)
        if vl_recall > best_recall:
            best_recall = vl_recall
            best_state  = {k: v.clone() for k, v in m.state_dict().items()}

    m.load_state_dict(best_state)
    _, _, labels, probs, _ = run_epoch(te_loader, m, crit)
    preds   = (probs >= OPTIMAL_THRESHOLD).astype(int)
    metrics = compute_metrics(labels, preds, probs)
    print(f"  Recall: {metrics['Recall']:.4f}  F2: {metrics['F2']:.4f}  "
          f"Precision: {metrics['Precision']:.4f}  AUC-ROC: {metrics['AUC-ROC']:.4f}")
    return metrics


In [ ]:
# ── Run cross-domain experiments ──────────────────────────────────────────────
results_b1 = cross_domain_run(train_domain="Walls", test_domain="Decks")
results_b2 = cross_domain_run(train_domain="Decks", test_domain="Walls")

# ── Compare against intra-domain performance ──────────────────────────────────
cross_domain_summary = pd.DataFrame({
    "Train → Test": ["Walls → Walls (intra)", "Decks → Decks (intra)",
                     "Walls → Decks (cross)", "Decks → Walls (cross)"],
    "Recall":    [df_metrics.loc["Walls", "Recall"],    df_metrics.loc["Decks", "Recall"],
                  results_b1["Recall"], results_b2["Recall"]],
    "F2":        [df_metrics.loc["Walls", "F2"],        df_metrics.loc["Decks", "F2"],
                  results_b1["F2"],     results_b2["F2"]],
    "Precision": [df_metrics.loc["Walls", "Precision"], df_metrics.loc["Decks", "Precision"],
                  results_b1["Precision"], results_b2["Precision"]],
}).set_index("Train → Test").round(4)

print("\nCross-Domain Summary:")
print(cross_domain_summary.to_string())


### 8.1 Cross-Domain Findings

*(Complete after running the cells above.)*

**Interpreting the results:**
- If cross-domain recall drops significantly (>10 percentage points) vs intra-domain recall, the model is learning texture, not crack morphology — domain-specific models should be recommended to the client.
- If cross-domain recall holds up, the model has generalised to crack morphology — a single generic model is defensible.
- Compare this result against the Phase 1 PCA finding: if Walls and Decks were clearly separable in PCA space, a large recall drop is expected and confirms the PCA prediction.

**Client recommendation:** Based on cross-domain recall, make a clear recommendation in the Phase 3 report — single generic model or separate models per asset type. Justify with these numbers.